# 04 — Model 2: Failure Type Prediction

**Goal:** after a failure is identified, predict which failure types are present.

This is a **multi-label** problem because one failed machine can have more than one failure type.

In [1]:
import os
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

DATA_PATH = "../data/predictive_maintenance.csv"
MODEL_DIR = "../models"
OUT_DIR = "../outputs/model_failure_types"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
feature_cols = ["Type", "Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
labels = ["TWF", "HDF", "PWF", "OSF"]

failure_df = df[df["Machine failure"] == 1].copy()
X_failure = failure_df[feature_cols].copy()
Y_failure = failure_df[labels].copy()

s1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(s1.split(X_failure, Y_failure))
X_train_val, Y_train_val = X_failure.iloc[train_val_idx].copy(), Y_failure.iloc[train_val_idx].copy()
X_test, Y_test = X_failure.iloc[test_idx].copy(), Y_failure.iloc[test_idx].copy()

s2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, val_idx = next(s2.split(X_train_val, Y_train_val))
X_train, Y_train = X_train_val.iloc[train_idx].copy(), Y_train_val.iloc[train_idx].copy()
X_val, Y_val = X_train_val.iloc[val_idx].copy(), Y_train_val.iloc[val_idx].copy()

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["Type"]),
    ("num", StandardScaler(), ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"])
])

X_train_p = preprocessor.fit_transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)


In [2]:
model2 = OneVsRestClassifier(
    RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
)
model2.fit(X_train_p, Y_train)
print("Model 2 trained successfully.")


Model 2 trained successfully.


In [3]:
Y_val_pred = model2.predict(X_val_p)
print("=== Validation Report ===")
print(classification_report(Y_val, Y_val_pred, target_names=labels, zero_division=0, digits=4))

Y_test_pred = model2.predict(X_test_p)
print("=== Final Test Report ===")
print(classification_report(Y_test, Y_test_pred, target_names=labels, zero_division=0, digits=4))


=== Validation Report ===
              precision    recall  f1-score   support

         TWF     0.8000    1.0000    0.8889         8
         HDF     0.7826    0.9474    0.8571        19
         PWF     0.8333    1.0000    0.9091        15
         OSF     0.9333    0.8750    0.9032        16

   micro avg     0.8333    0.9483    0.8871        58
   macro avg     0.8373    0.9556    0.8896        58
weighted avg     0.8397    0.9483    0.8877        58
 samples avg     0.8697    0.9455    0.8909        58

=== Final Test Report ===
              precision    recall  f1-score   support

         TWF     1.0000    1.0000    1.0000         9
         HDF     0.9091    0.8696    0.8889        23
         PWF     0.8571    0.9474    0.9000        19
         OSF     0.9000    0.9000    0.9000        20

   micro avg     0.9028    0.9155    0.9091        71
   macro avg     0.9166    0.9292    0.9222        71
weighted avg     0.9042    0.9155    0.9091        71
 samples avg     0.8676  

In [4]:
import json
from sklearn.metrics import precision_score, recall_score, f1_score

metrics = {
    "micro_precision": precision_score(
        Y_test, Y_test_pred, average="micro", zero_division=0
    ),
    "micro_recall": recall_score(
        Y_test, Y_test_pred, average="micro", zero_division=0
    ),
    "micro_f1": f1_score(
        Y_test, Y_test_pred, average="micro", zero_division=0
    ),
    "macro_precision": precision_score(
        Y_test, Y_test_pred, average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        Y_test, Y_test_pred, average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        Y_test, Y_test_pred, average="macro", zero_division=0
    )
}

joblib.dump(
    model2,
    os.path.join(MODEL_DIR, "model_failure_types_rf.joblib")
)

joblib.dump(
    preprocessor,
    os.path.join(MODEL_DIR, "model_failure_types_preprocessor.joblib")
)

joblib.dump(
    {
        "feature_cols": feature_cols,
        "labels": labels
    },
    os.path.join(MODEL_DIR, "model_failure_types_metadata.joblib")
)

with open(
    os.path.join(OUT_DIR, "metrics.json"), "w"
) as f:
    json.dump(metrics, f, indent=2)

print("Saved Model 2 artifacts and metrics.")

Saved Model 2 artifacts and metrics.
